# 执行模型与异步调度

学习目标：能依据运行上下文解释调用顺序，并区分语言作业、宿主队列和跨 Realm 对象。

前置知识：函数调用、闭包、Promise、async/await 和模块模式。

适用版本：ECMAScript 2025、Node.js 24.11.0；.mjs 使用 ES 模块。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/23-execution-model/。

1. [stack.mjs](scripts/23-execution-model/stack.mjs)：调用与返回的栈顺序。
2. [index.html](scripts/23-execution-model/index.html)：最小浏览器交互页面。
3. [browser.mjs](scripts/23-execution-model/browser.mjs)：点击事件中的任务与微任务。
4. [order.cjs](scripts/23-execution-model/order.cjs)：CommonJS 顶层调度。
5. [order.mjs](scripts/23-execution-model/order.mjs)：ESM 顶层调度。
6. [yield.mjs](scripts/23-execution-model/yield.mjs)：分批计算与宿主让出。
7. [realm.mjs](scripts/23-execution-model/realm.mjs)：不同内置对象身份。

Step 1：运行调用栈示例。

```bash
node scripts/23-execution-model/stack.mjs
```

Step 2：运行 CommonJS 顶层示例。

```bash
node scripts/23-execution-model/order.cjs
```

Step 3：运行 ESM 顶层示例。

```bash
node scripts/23-execution-model/order.mjs
```

Step 4：运行分批计算。

```bash
node scripts/23-execution-model/yield.mjs
```

Step 5：运行跨 Realm 判断。

```bash
node scripts/23-execution-model/realm.mjs
```

## 1 执行上下文、调用栈与运行至完成

执行上下文（execution context）是规范用于描述当前代码、词法环境与恢复位置的机制，不是必须能直接读取的 JavaScript 对象。普通函数调用进入新上下文，返回时恢复调用者；这些活动上下文形成栈。

一个 Agent 内同时只有一个活动执行上下文。同一段同步工作不会被 Promise 回调从中间打断，这通常称为运行至完成（run to completion）。await、yield 等有规范定义的暂停点。运行至完成不意味着整个程序只能有一次任务，也不意味着所有宿主工作都在同一线程。

配套 [stack.mjs](scripts/23-execution-model/stack.mjs)：

```javascript
const events = [];
function inner() { events.push("inner"); }
function outer() {
  events.push("outer before");
  Promise.resolve().then(() => {
    events.push("reaction");
    console.log(events.join(",")); // → outer before,inner,outer after,script after,reaction
  });
  inner();
  events.push("outer after");
}
outer();
events.push("script after");
```

## 2 语言作业与浏览器事件循环

ECMAScript 的作业（job）描述以后执行的入口，例如 Promise 反应；宿主通过接口安排它。HTML 事件循环再规定任务（task）、微任务（microtask）及渲染机会。多个任务队列不等同于一个固定全局 FIFO 队列；微任务检查点会处理队列中新追加的微任务。

同步代码先结束，已排入队列的 Promise 处理和 queueMicrotask 回调在微任务检查点运行，定时器回调属于后续任务。setTimeout 的延迟是调度条件，不是精确开始时刻。持续追加微任务会延后任务和渲染；await Promise.resolve() 也不能保证浏览器已经绘制。

以下页面只使用浏览器 DOM 来触发一次观察；每次点击先禁用按钮，等本次定时器完成再启用，防止多次实验的输出混合。按 README.md 启动服务，打开[本地调度页面](http://127.0.0.1:8102/scripts/23-execution-model/index.html)，点击“运行一次”并查看页面及 Console。结束后按 Ctrl+C 停止服务。

配套 [index.html](scripts/23-execution-model/index.html)：

```html
<!doctype html>
<html lang="zh-CN">
<meta charset="utf-8">
<title>任务与微任务</title>
<button id="run">运行一次</button>
<pre id="output">等待运行</pre>
<script type="module" src="./browser.mjs"></script>
</html>
```

配套 [browser.mjs](scripts/23-execution-model/browser.mjs)：

```javascript
const button = document.querySelector("#run");
const output = document.querySelector("#output");
button.addEventListener("click", () => {
  button.disabled = true;
  const events = ["sync start"];
  Promise.resolve().then(() => events.push("promise"));
  queueMicrotask(() => events.push("microtask"));
  setTimeout(() => {
    events.push("timer");
    output.textContent = events.join("\n");
    console.log(events.join(",")); // → sync start,sync end,promise,microtask,timer
    button.disabled = false;
  }, 0);
  events.push("sync end");
});
```

## 3 Node.js 的队列与模块上下文

Node.js 的事件循环有 timers、poll、check 等阶段；setImmediate 进入 check 阶段，I/O 回调由相应宿主阶段处理。不要把浏览器渲染步骤套入 Node.js，也不要假定顶层 setTimeout(0) 与 setImmediate 有跨环境固定的先后。

process.nextTick 是 Node.js 专有队列。下面两个文件故意放在各自顶层：CommonJS 初始脚本结束后先处理 nextTick，再处理微任务；ESM 初始求值已处于微任务处理过程，所登记的 Promise 与 queueMicrotask 先于 nextTick。把同样代码移入 I/O 回调后，不能继续机械套用“ESM 顶层顺序”。

配套 [order.cjs](scripts/23-execution-model/order.cjs)：

```javascript
const events = ["sync"];
process.nextTick(() => events.push("nextTick"));
Promise.resolve().then(() => events.push("promise"));
queueMicrotask(() => events.push("microtask"));
setImmediate(() => console.log(events.join(","))); // → sync,nextTick,promise,microtask
```

配套 [order.mjs](scripts/23-execution-model/order.mjs)：

```javascript
const events = ["sync"];
process.nextTick(() => events.push("nextTick"));
Promise.resolve().then(() => events.push("promise"));
queueMicrotask(() => events.push("microtask"));
setImmediate(() => console.log(events.join(","))); // → sync,promise,microtask,nextTick
```

## 4 阻塞与让出执行机会

长同步循环占用当前 Agent 的执行机会，Promise 不会自动让计算移到后台。拆分计算并在批次间等待 Node.js 的 setImmediate，能让事件循环处理其他回调；批次大小应由实际响应时间与调度成本选择。

本例把总数 1000 分成每批 250 项。它验证别的回调在计算尚未全部完成时得到执行机会，不根据任意毫秒延迟猜测任务是否完成。CPU 密集任务的真正并行属于 Worker 等运行时专题。

配套 [yield.mjs](scripts/23-execution-model/yield.mjs)：

```javascript
import assert from "node:assert/strict";
import { setImmediate as yieldTurn } from "node:timers/promises";
let processed = 0;
let observed;
setImmediate(() => { observed = processed; });
let total = 0;
for (let start = 1; start <= 1000; start += 250) {
  for (let value = start; value < start + 250; value += 1) {
    total += value;
    processed += 1;
  }
  await yieldTurn();
}
assert.equal(total, 500500);
assert.equal(observed, 250);
console.log(total, "callback after", observed); // → 500500 callback after 250
```

## 5 Realm、Agent 与跨环境判断

Realm 包含一组内置对象及全局环境；不同 Realm 可以拥有不同的 Array 构造器和 Array.prototype。Agent 是执行及作业处理的单位，可以包含多个 Realm；跨 Realm 不等于多线程。多个 Agent 的共享内存交互另由内存模型约束。

instanceof 通常沿原型链检查特定构造器的 prototype，因此跨 Realm 的数组未必是当前 Array 的实例。Array.isArray 检查数组身份，适合此处。Node.js 的 vm 可创建独立上下文来演示这种差异；它不是运行不可信代码的安全隔离机制。

配套 [realm.mjs](scripts/23-execution-model/realm.mjs)：

```javascript
import assert from "node:assert/strict";
import vm from "node:vm";
const foreign = vm.runInNewContext("[10, 20]");
assert.equal(foreign instanceof Array, false);
assert.equal(Array.isArray(foreign), true);
console.log(foreign instanceof Array, Array.isArray(foreign)); // → false true
```

## 本章小结

- 调用栈决定当前同步执行，作业的排入与宿主队列决定后续执行。
- 比较顺序必须保留宿主、模块模式和回调位置条件。
- Realm 的内置对象身份影响 instanceof；Agent 与 Realm 是不同层次。

## 练习

1. 交换两种模块中 Promise 与 queueMicrotask 的登记顺序；标准：它们的相对输出随登记顺序交换，nextTick 的上下文差异仍存在。
2. 将分批大小改为 100；标准：总和仍为 500500，首次回调观察到 processed 为 100。
3. 连续运行浏览器示例两次；标准：两次都只有五行，无交叉残留，解释为什么微任务不等于一次绘制。

## 参考与引用来源

- TC39（ECMA-262 第 16 版）：[§9.3–9.7 Realm、执行上下文、Jobs 与 Agents](https://tc39.es/ecma262/2025/multipage/executable-code-and-execution-contexts.html#sec-execution-contexts)、[§27.2.2 Promise Jobs](https://tc39.es/ecma262/2025/multipage/control-abstraction-objects.html#sec-promise-jobs)。
- WHATWG：[HTML 事件循环处理模型和微任务检查点](https://html.spec.whatwg.org/multipage/webappapis.html#event-loop-processing-model)：浏览器任务、微任务与渲染机会。
- Node.js：[事件循环阶段](https://nodejs.org/en/learn/asynchronous-work/event-loop-timers-and-nexttick)；24.11.0 [nextTick 与 queueMicrotask](https://nodejs.org/download/release/v24.11.0/docs/api/process.html#when-to-use-queuemicrotask-vs-processnexttick)、[定时器](https://nodejs.org/download/release/v24.11.0/docs/api/timers.html)、[vm.runInNewContext](https://nodejs.org/download/release/v24.11.0/docs/api/vm.html#vmruninnewcontextcode-contextobject-options)：宿主调度与上下文示例。